# Experiment 1: Time and Frequency Domain, and the Pendulum
**Course:** CSAI4014 Algorithms for Intelligent Systems and Robotics  
**Student:** Aadi Chopra | SAP ID: 500120163 | Roll No: R2142231902 | Batch: B2 AIML  

### 1.1 Aim
To represent a sampled signal in the time domain and in the frequency domain using the discrete Fourier transform, and to simulate the motion of a simple pendulum by numerical integration of its equation of motion.


In [ ]:
import numpy as np
import scipy.integrate as integrate
import matplotlib.pyplot as plt

# Sampling parameters
fs = 500.0  # sampling rate (Hz)
T = 2.0     # record length (seconds)
dt = 1.0 / fs
t = np.arange(0, T, dt)
N = len(t)
df = fs / N
print(f"N samples: {N}, dt: {dt*1000:.2f} ms, df: {df:.2f} Hz, Nyquist: {fs/2:.1f} Hz")

# Synthesize signal with 5, 20, 50 Hz plus Gaussian noise
rng = np.random.default_rng(0)
x = (1.00 * np.sin(2 * np.pi * 5 * t) + 
     0.50 * np.sin(2 * np.pi * 20 * t) + 
     0.25 * np.sin(2 * np.pi * 50 * t) + 
     0.20 * rng.standard_normal(N))


In [ ]:
# Compute one-sided amplitude spectrum
X = np.fft.rfft(x)
f = np.fft.rfftfreq(N, d=dt)
amp = np.abs(X) * 2.0 / N
amp[0] /= 2.0

# Print dominant peak amplitudes
peak_idx = [int(5.0/df), int(20.0/df), int(50.0/df)]
for idx in peak_idx:
    print(f"Frequency: {f[idx]:.1f} Hz -> Measured Amplitude: {amp[idx]:.3f}")


In [ ]:
# Aliasing demonstration: 5 Hz sampled at 8 Hz
fs_low = 8.0
t_low = np.arange(0, T, 1.0/fs_low)
N_low = len(t_low)
x_low = np.sin(2 * np.pi * 5 * t_low)
X_low = np.fft.rfft(x_low)
f_low = np.fft.rfftfreq(N_low, d=1.0/fs_low)
amp_low = np.abs(X_low) * 2.0 / N_low

aliased_idx = np.argmax(amp_low[1:]) + 1
print(f"5 Hz sampled at 8 Hz -> Aliased Peak at: {f_low[aliased_idx]:.2f} Hz (Expected |8 - 5| = 3.0 Hz)")


In [ ]:
# Pendulum numerical integration
g, L = 9.81, 1.0
T0 = 2 * np.pi * np.sqrt(L / g)
print(f"Small-angle theoretical period T0: {T0:.4f} s")

def pendulum_deriv(t, state, b):
    theta, omega = state
    return [omega, -b * omega - (g / L) * np.sin(theta)]

# Sweep damping coefficients
t_span = (0, 10)
t_eval = np.linspace(0, 10, 1000)
theta0 = np.radians(40)
init_state = [theta0, 0.0]

sol_undamped = integrate.solve_ivp(pendulum_deriv, t_span, init_state, args=(0.0,), t_eval=t_eval)
sol_light = integrate.solve_ivp(pendulum_deriv, t_span, init_state, args=(0.3,), t_eval=t_eval)
sol_heavy = integrate.solve_ivp(pendulum_deriv, t_span, init_state, args=(2.0,), t_eval=t_eval)
print("Integration complete for damping b = 0, 0.3, 2.0")
